# LELA60331 Week 8 Seminar: Turning text into features

Last week we finished building logistic regression on simulated data, where each text had just two features. Now we move to **real text**: 10,000 Yelp reviews. Before we can train a classifier on them, we have to turn each review into numbers. In this sheet we:

| Part | What you do |
|---|---|
| 1 | Load the Yelp reviews and get to know them |
| 2 | Build a vocabulary of the 5000 most frequent words |
| 3 | Split the reviews into a training set and a test set |
| 4 | One-hot encode the reviews: one feature for every word in the vocabulary |
| 5 | Write matrix-vector multiplication for these one-hot features |

Each step is done twice: first on a **tiny example** that you can check by hand, then on the real reviews. As before, each activity has a cell to complete (replace every `???`) followed by a ✅ **check** cell. When you see a 🔮 **Predict** prompt, decide what you think will happen *before* you run the cell.

In [ ]:
# ▶ Run this cell first. It defines the check() helper used after each activity.
# You don't need to understand how it works: it prints ✅ when your answer is right and ❌ when it isn't yet.
def check(label, condition):
    try:
        passed = bool(condition() if callable(condition) else condition)
    except Exception as err:
        passed = False
        label += f"  (error: {type(err).__name__}: {err})"
    print(("✅ " if passed else "❌ ") + label)
    return passed

---
# Part 1 · The Yelp reviews

Our data set is 10,000 reviews from Yelp, each labelled **negative** (1 or 2 stars) or **positive** (3 or 4 stars).

### Step 1 · Download the file

A line beginning with `!` is a command for the computer, not Python: `wget` downloads a file from the web. Run the cell.

In [ ]:
!wget https://raw.githubusercontent.com/cbannard/lela60331_24-25/refs/heads/main/data/yelp_reviews.txt

### Step 2 · Look at the file

The file is **tab-separated**: each line holds a review, then a tab, then the label. The first line is a header. `head -n2` shows the first two lines:

In [ ]:
!head -n2 yelp_reviews.txt

The text has already been lower-cased and its punctuation separated from the words by spaces, so that it is easy to split into words. (The odd `n n` is what is left of the line breaks.)

### Step 3 · Read it into Python

This cell reads the file into two lists: `reviews` holds the texts and `labels` the labels, so review `i` has the label `labels[i]`. Just run it.

In [ ]:
reviews = []
labels = []

with open("yelp_reviews.txt") as fp:
    lines = fp.readlines()
for line in lines[1:]:                         # skip the header line
    fields = line.rstrip().split("\t")         # [review, label]
    reviews.append(fields[0])
    labels.append(fields[1])

print(labels[:5])                              # the first five labels
print(reviews[0][:100])                        # the first 100 characters of review 0

### Activity 1 — How many reviews?

1. `n_reviews`: the number of reviews.
2. `n_positive_reviews`: the number of positive reviews. *Hint:* the labels are strings this time (`"positive"` and `"negative"`), so use the list method `count`.

In [ ]:
n_reviews = ???                                                # 1.
n_positive_reviews = ???                                       # 2.
print(n_reviews, n_positive_reviews)                           # you should get 10000 5000

In [ ]:
# ✅ Check
check("1. n_reviews", lambda: n_reviews == 10000)
check("2. n_positive_reviews", lambda: n_positive_reviews == 5000)

Are the classes balanced? What accuracy would a model get if it always said "positive"?

### Activity 2 — Looking at one review

1. `first_words`: review 0 split into a list of words. *Hint:* split it at the spaces.
2. `first_review_length`: the number of words in review 0.

Then read review 0. Does its label make sense?

In [ ]:
first_words = ???                                              # 1.
print(first_words[:10])                                        # the first 10 words
first_review_length = ???                                      # 2.
print(first_review_length)                                     # you should get 118
print(labels[0], ":", reviews[0])

In [ ]:
# ✅ Check
check("1. first_words", lambda: first_words == reviews[0].split())
check("2. first_review_length", lambda: first_review_length == 118)

---
# Part 2 · Building a vocabulary

Our features will be the words. Each review gets one feature for every word in a **vocabulary**: 1 if the word occurs in the review and 0 if it doesn't. We use the 5000 most frequent words as our vocabulary (not an ideal solution, but a convenient one).

We build it in three steps: split each review into words, count the words, and keep the 5000 most frequent.

### Step 1 · Tokenise

First we **tokenise** each review, splitting it into a list of words, just as you did for review 0 in Activity 2. This is a very naive tokenisation that just splits the text at spaces. Run the cell.

In [ ]:
tokenized_sents = []
for review in reviews:
    tokenized_sents.append(review.split())

print(len(tokenized_sents))         # one list of words per review
print(tokenized_sents[0][:12])      # the first 12 words of review 0

### Step 2 · Count the words

We count the words with a **counting dictionary** (Week 3 seminar): each word is a key and its value is the number of times the word occurs. `counts.get(word, 0)` gives the count so far, or 0 if the word hasn't been seen yet.

### Activity 3 — Counting the words of one sentence

Complete the line in the loop: the word's new count is its count so far plus 1.

🔮 **Predict** first: what will the count of `"good"` be?

In [ ]:
tiny_counts = {}
for word in "the food was good and the service was good".split():
    tiny_counts[word] = ???

print(tiny_counts)      # you should get {'the': 2, 'food': 1, 'was': 2, 'good': 2, 'and': 1, 'service': 1}

In [ ]:
# ✅ Check
check("tiny_counts", lambda: tiny_counts == {'the': 2, 'food': 1, 'was': 2, 'good': 2, 'and': 1, 'service': 1})

### Activity 4 — Counting the words of all the reviews

Now do the same for every review. There are two loops this time: one over the reviews, and one over the words in each review. The line inside is the same as in Activity 3, but with `counts`.

In [ ]:
counts = {}
for sent in tokenized_sents:            # each review
    for word in sent:                   # each word in that review
        counts[word] = ???

print(len(counts))                              # the number of different words: you should get 28495
print(counts["good"], counts["bad"])            # you should get 5012 1285

In [ ]:
# ✅ Check
check("number of different words", lambda: len(counts) == 28495)
check("count of 'good'", lambda: counts["good"] == 5012)
check("count of 'bad'", lambda: counts["bad"] == 1285)

### Step 3 · The 5000 most frequent words

To find the most frequent words, we make a list of `(count, word)` pairs and sort it from largest to smallest, as in the Week 3 seminar. Python sorts pairs by their first element, the count. Run the cell.

In [ ]:
pairs = [(count, word) for word, count in counts.items()]
pairs = sorted(pairs, reverse=True)
print(pairs[:5])

### Activity 5 — Keeping just the words

We only need the **words** from the pairs. In a list comprehension, `for count, word in ...` **unpacks** each pair into its two parts.

1. `tiny_words`: the words from `tiny_pairs`, a short list of pairs, using a list comprehension that unpacks each pair into `count, word`
2. `top_pairs`: the first 5000 pairs in `pairs`
3. `type_list`: the words from `top_pairs`, in the same way as step 1

In [ ]:
tiny_pairs = [(5, "the"), (3, "food"), (1, "good")]
tiny_words = ???                                               # 1.
print(tiny_words)                                              # you should get ['the', 'food', 'good']

top_pairs = ???                                                # 2.
type_list = ???                                                # 3.
print(len(type_list))                                          # you should get 5000
print(type_list[:10])                                          # the 10 most frequent words

In [ ]:
# ✅ Check
check("1. tiny_words", lambda: tiny_words == ["the", "food", "good"])
check("2. top_pairs", lambda: top_pairs == pairs[:5000])
check("3. 5000 words, the most frequent first", lambda: len(type_list) == 5000 and type_list[0] == pairs[0][1])
check("3. only words, not pairs", lambda: all(type(word) == str for word in type_list))

What kinds of words are most frequent? Do you think they will be useful for telling positive from negative reviews?

### Step 4 · From a word to its column

Each word in the vocabulary gets its own feature, and so its own **column** in our data matrix: `type_list[j]` is the word for column `j`. We also need to go the other way, from a word to its column, so we make a dictionary `word_to_index`.

🔮 **Predict**: the cell prints `word_to_index["good"]`, and then `type_list` at that position. What will the second one be? Run the cell.

In [ ]:
word_to_index = {}
for j in range(len(type_list)):
    word_to_index[type_list[j]] = j

print(word_to_index["good"], type_list[word_to_index["good"]])

---
# Part 3 · Training and test sets

We want to know how well our classifier does on reviews it has **never seen**. So we split the data: we train on 80% of the reviews (8000) and test on the other 20% (2000). In a real project we would also keep a third set, a **development** set, for trying out different settings; here we just use training and test.

### Step 1 · A tiny split

`random.sample(sequence, k)` picks `k` different elements at random from `sequence`. Here it picks 8 of the numbers 0–9 for "training"; the "test" numbers are the ones that were not picked.

🔮 **Predict**: how many test numbers will there be? Run the cell (more than once, if you like: the numbers change each time).

In [ ]:
import random

tiny_train = random.sample(range(10), 8)
tiny_test = [i for i in range(10) if i not in tiny_train]
print(tiny_train, tiny_test)

### Step 2 · The real split

Now the same for the 10,000 reviews: we pick the indices of 8000 training reviews, and the test reviews are all the indices that were not picked. Run the cell.

In [ ]:
random.seed(1)        # so that everyone gets the same split
train_ints = random.sample(range(len(reviews)), int(len(reviews) * 0.8))

# The test items are all the indices that were not picked for training.
# Checking whether something is in a set is very fast.
train_set = set(train_ints)
test_ints = []
for i in range(len(reviews)):
    if i not in train_set:
        test_ints.append(i)

print(len(train_ints), len(test_ints))
print(train_ints[:5])

### Activity 6 — Labels as numbers

Our model works with labels 1 (positive) and 0 (negative), as in Week 5. `int(labels[i] == "positive")` is 1 if review `i` is positive and 0 if not, because `int(True)` is 1 and `int(False)` is 0.

1. `first_train_label`: the label, as 1 or 0, of the **first training review**. *Hint:* its index is `train_ints[0]`.
2. `y_train`: the label of each training review, as 1 or 0, in the order of `train_ints`. *Hint:* a list comprehension over `train_ints`.
3. `y_test`: the same for the test reviews

In [ ]:
first_train_label = ???                                                   # 1.
print(train_ints[0], labels[train_ints[0]], first_train_label)     # you should get 2201 negative 0

y_train = ???                                                             # 2.
y_test = ???                                                              # 3.
print(len(y_train), sum(y_train))       # you should get 8000 4026
print(len(y_test), sum(y_test))         # you should get 2000 974

In [ ]:
# ✅ Check
check("1. first_train_label", lambda: first_train_label == int(labels[train_ints[0]] == "positive"))
check("2. y_train", lambda: len(y_train) == 8000 and sum(y_train) == 4026)
check("3. y_test", lambda: len(y_test) == 2000 and sum(y_test) == 974)
check("no review is in both sets", lambda: len(set(train_ints) & set(test_ints)) == 0)

---
# Part 4 · One-hot encoding

Now we turn each review into numbers. We want a matrix **M_train** with one row per training review (8000) and one column per vocabulary word (5000), and a matrix **M_test** in the same way for the 2000 test reviews. Element $x_{i,j}$ is 1 if word $j$ occurs in review $i$, and 0 otherwise. This is called **one-hot encoding**.

### Sparse rows

As a list of lists, M_train would hold 8000 × 5000 = 40 million numbers. That is very slow to work with in plain Python, and almost all of them are 0: a review contains only a small fraction of the 5000 vocabulary words. So instead we use a **sparse** representation: for each review we store only the **columns that hold a 1**, as a list of column numbers.

| | `good` (col 0) | `bad` (col 1) | `food` (col 2) | `service` (col 3) | sparse row |
|---|---|---|---|---|---|
| "good food good" | 1 | 0 | 1 | 0 | `[0, 2]` |

### Activity 7 — A tiny example: the full row

Here is a vocabulary of just 4 words and the review from the table. Complete the condition: column `j` should hold a 1 if the word `tiny_vocab[j]` is in the review.

In [ ]:
tiny_vocab = ["good", "bad", "food", "service"]      # columns 0, 1, 2 and 3
review = "good food good".split()

full_row = [0, 0, 0, 0]
for j in range(len(tiny_vocab)):
    if ???:
        full_row[j] = 1
print(full_row)      # you should get [1, 0, 1, 0]

In [ ]:
# ✅ Check
check("full_row", lambda: full_row == [1, 0, 1, 0])

"good" occurs twice in the review, but its column holds a 1, not a 2: we only record whether a word is **present**.

### Activity 8 — A tiny example: the sparse row

Now make the sparse row instead: rather than putting a 1 in column `j`, add the **column number** `j` to the list. The condition is the same as in Activity 7.

In [ ]:
sparse_row = []
for j in range(len(tiny_vocab)):
    if ???:
        sparse_row.append(???)
print(sparse_row)    # you should get [0, 2]

In [ ]:
# ✅ Check
check("sparse_row", lambda: sparse_row == [0, 2])

### Why the sparse row is enough

The dot product of a row of 0s and 1s with a vector adds up the vector's elements wherever the row has a 1, and ignores the rest (they are multiplied by 0). The sparse row lists exactly those positions.

To see this, we need `dot` and `mat_vec` from Week 7. Run the cell to define them, and compare the two results.

In [ ]:
def dot(a, b):
    # the dot product of two lists of the same length
    if len(a) != len(b):
        raise ValueError("the two lists must have the same length")
    total = 0
    for i in range(len(a)):
        total += a[i] * b[i]
    return total

def mat_vec(matrix, vector):
    # matrix-vector multiplication: the dot product of each row of the matrix with the vector (Week 7, Part 4)
    result = []
    for row in matrix:
        result.append(dot(row, vector))
    return result


tiny_weights = [0.5, -2.0, 0.25, 1.0]

total = 0
for j in sparse_row:
    total += tiny_weights[j]
print(dot(full_row, tiny_weights), total)       # both 0.75: 0.5 (good) + 0.25 (food)

### Step 2 · One real review

For a real review, it is quicker to go through the **words of the review** than through all 5000 vocabulary words, and to look up each word's column in `word_to_index`.

A word can occur several times in a review, but its column should only be stored once. So we first collect the columns in a **set**, which keeps only one copy of each element. Run the cell to see how a set behaves.

In [ ]:
s = set()
for j in [7, 2, 7, 5, 2]:
    s.add(j)
print(s)              # each number only once
print(sorted(s))      # sorted() turns the set into a sorted list

### Activity 9 — One-hot encoding one review

Encode review 0. For each word in the review, fill in:

1. the condition: only words that are in our vocabulary have a column (`word_to_index` contains exactly those words)
2. the column for that word

In [ ]:
first_tokens = tokenized_sents[0]
first_row = set()
for word in first_tokens:
    if ???:                    # 1.
        first_row.add(???)       # 2.
first_row = sorted(first_row)

print(len(first_tokens), "words;", len(first_row), "different vocabulary words")    # you should get 118 and 72
print(first_row[:10])                              # the first 10 columns that hold a 1
print([type_list[j] for j in first_row[:10]])      # ... and their words

In [ ]:
# ✅ Check
check("1–2. first_row", lambda: first_row == sorted(set(word_to_index[w] for w in tokenized_sents[0] if w in word_to_index)))
check("each column stored once", lambda: len(first_row) == len(set(first_row)))

### Activity 10 — One-hot encoding all the reviews

We need to do this for every review, and to build two matrices (training and test), so we put the steps in a **function**. `one_hot(token_lists)` takes a list of tokenised reviews and returns their sparse matrix: a list with one row per review.

The inside of the loop is exactly what you wrote in Activity 9: fill in the same two blanks.

In [ ]:
def one_hot(token_lists):
    M = []
    for tokens in token_lists:               # each review
        row = set()
        for word in tokens:
            if ???:              # 1.
                row.add(???)       # 2.
        M.append(sorted(row))                # the columns that hold a 1, in order
    return M

print(one_hot([first_tokens]) == [first_row])       # the same as Activity 9: you should get True

In [ ]:
# ✅ Check
check("one_hot gives the same as Activity 9", lambda: one_hot([first_tokens]) == [first_row])
check("one_hot with two tiny reviews", lambda: one_hot([["good", "good"], ["xyzzy", "bad"]]) == [[word_to_index["good"]], [word_to_index["bad"]]])

Now use the function on the training and test reviews. First we collect the tokenised reviews for each set, in the order of `train_ints` and `test_ints`; then we encode them. Encoding takes a few seconds. Run the cell.

In [ ]:
train_tokens = [tokenized_sents[i] for i in train_ints]
test_tokens = [tokenized_sents[i] for i in test_ints]

M_train = one_hot(train_tokens)
M_test = one_hot(test_tokens)
print(len(M_train), len(M_test))      # the number of rows (reviews) in each matrix: 8000 2000
print(M_train[0][:10])                # the first 10 columns that hold a 1 in the first training review
print([type_list[j] for j in M_train[0][:10]])     # ... and their words

In [ ]:
# ✅ Check
check("8000 training rows and 2000 test rows", lambda: len(M_train) == 8000 and len(M_test) == 2000)
check("the first training review",
      lambda: M_train[0] == sorted(set(word_to_index[w] for w in train_tokens[0] if w in word_to_index)))
check("each column stored once, in order", lambda: all(row == sorted(set(row)) for row in M_train[:100]))
check("every column is between 0 and 4999", lambda: all(0 <= j < 5000 for row in M_test for j in row))

### Activity 11 — How sparse?

1. `ones_per_row`: a list with the number of 1s in each row of `M_train` (the number of different vocabulary words in each training review). *Hint:* a list comprehension with `len`.
2. `average_ones`: the average of `ones_per_row`

What proportion of the 5000 columns is that?

In [ ]:
ones_per_row = ???                                            # 1.
print(ones_per_row[:5])
average_ones = ???                                            # 2.
print(round(average_ones, 1))                   # you should get 78.5
print(round(average_ones / 5000, 3))            # the proportion of columns that hold a 1

In [ ]:
# ✅ Check
check("1. ones_per_row", lambda: len(ones_per_row) == 8000 and ones_per_row[0] == len(M_train[0]))
check("2. average_ones", lambda: abs(average_ones - 78.5) < 0.01)

---
# Part 5 · Matrix-vector multiplication with sparse matrices

At the end of Week 7 we saw that training needs two matrix-vector multiplications:

* **matrix × weights** → a score for every review
* **transpose × errors** → a gradient for every weight

We now write both for our sparse matrices, starting each time with one row by hand.

## Scores: sparse matrix × vector

To get the score of a sparse row, add up the weights at the columns that hold a 1, as in Part 4:

| sparse row | words | score with `tiny_weights = [0.5, -2.0, 0.25, 1.0]` |
|---|---|---|
| `[0, 2]` | good, food | 0.5 + 0.25 = **0.75** |
| `[1, 2]` | bad, food | −2.0 + 0.25 = **−1.75** |
| `[3]` | service | **1.0** |

### Activity 12 — The score of one row

Complete the line in the loop, so that `total` is the score of the row `[1, 2]`.

In [ ]:
row = [1, 2]
total = 0
for j in row:
    total += ???
print(total)          # you should get -1.75

In [ ]:
# ✅ Check
check("the score of [1, 2]", lambda: total == -1.75)

### Activity 13 — The scores of every row: `sparse_mat_vec`

`sparse_mat_vec(M, vector)` does the same job as `mat_vec` in Week 7, but for a sparse matrix. For each row it does what you did in Activity 12. Complete the line inside the inner loop.

In [ ]:
def sparse_mat_vec(M, vector):
    result = []
    for row in M:                     # each row
        total = 0
        for j in row:                 # each column that holds a 1
            total += ???
        result.append(total)
    return result

print(sparse_mat_vec([sparse_row], tiny_weights))                    # you should get [0.75]
print(sparse_mat_vec([[0, 2], [1, 2], [3]], tiny_weights))           # the table above: [0.75, -1.75, 1.0]

In [ ]:
# ✅ Check
check("one row", lambda: sparse_mat_vec([sparse_row], tiny_weights) == [0.75])
check("three rows", lambda: sparse_mat_vec([[0, 2], [1, 2], [3]], tiny_weights) == [0.75, -1.75, 1.0])
check("same as mat_vec with full rows",
      lambda: sparse_mat_vec([[0, 2], [1, 2]], tiny_weights) == mat_vec([[1, 0, 1, 0], [0, 1, 1, 0]], tiny_weights))

## Gradients: transpose × vector

For the gradients, Week 5 (and Week 7) used `dot(x[0], errors)`: for each **feature**, multiply its value in each text by that text's error, and add up. With one-hot features the value is 1 or 0, so the gradient for word `j` is simply **the sum of the errors of the reviews that contain word `j`**.

Here is an example with two reviews, `[[0, 2], [1, 2]]`, whose errors are `[10, 1]`:

| column | reviews that contain it | gradient |
|---|---|---|
| 0 | review 0 | **10** |
| 1 | review 1 | **1** |
| 2 | reviews 0 and 1 | 10 + 1 = **11** |
| 3 | none | **0** |

We can calculate this by going through the reviews one at a time: start with a 0 for every column, then for each review, add its error to each of its columns.

### Activity 14 — The gradients, one review at a time

Complete the two lines: for each column `j` in the review, add the review's error to `result[j]`.

In [ ]:
result = [0, 0, 0, 0]

for j in [0, 2]:              # review 0, whose error is 10
    result[j] += ???
print(result)                 # you should get [10, 0, 10, 0]

for j in [1, 2]:              # review 1, whose error is 1
    result[j] += ???
print(result)                 # you should get [10, 1, 11, 0], as in the table

In [ ]:
# ✅ Check
check("the gradients", lambda: result == [10, 1, 11, 0])

### Activity 15 — The gradients for every column: `sparse_transpose_mat_vec`

`sparse_transpose_mat_vec(M, vector, num_columns)` does Activity 14 for any sparse matrix. It starts with a 0 for every column, then goes through the rows: for each column `j` that holds a 1 in row `i`, it adds `vector[i]` (row `i`'s error) to column `j`'s total. Complete the line inside the inner loop.

In [ ]:
def sparse_transpose_mat_vec(M, vector, num_columns):
    result = [0] * num_columns
    for i in range(len(M)):           # each row
        for j in M[i]:                # each column that holds a 1 in that row
            result[j] += ???
    return result

print(sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4))     # the same as Activity 14: [10, 1, 11, 0]

In [ ]:
# ✅ Check
check("the example", lambda: sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4) == [10, 1, 11, 0])
check("same as mat_vec with the transposed full rows",
      lambda: sparse_transpose_mat_vec([[0, 2], [1, 2]], [10, 1], 4) == mat_vec([[1, 0], [0, 1], [1, 1], [0, 0]], [10, 1]))

---
## ✏️ Summary

In this sheet you have turned 10,000 reviews into features a classifier can use:

1. **tokenised** each review into a list of words
2. built a **vocabulary** of the 5000 most frequent words, with a column for each
3. split the reviews into a **training set** and a **test set**
4. **one-hot encoded** each review, storing only the columns that hold a 1 (a **sparse** matrix)
5. written the two matrix-vector multiplications that training needs, for sparse matrices